# FACTS — Phase 2: Data Preprocessing
**Behafarin Emam | be379@drexel.edu**  
Capstone 2, Week 4 Coding Sample

---

## Overview

This notebook prepares the data for the revised FACTS methodology. The project detects LLM hallucinations using **self-consistency signals** — if a model truly knows the answer to a question, it should respond consistently regardless of sampling temperature. High inconsistency across temperature-varied outputs signals factual uncertainty and likely hallucination.

### Revised approach (from proposal)
The original proposal used GHI scores as supervised labels for an XGBoost classifier. This is circular: a classifier trained on GHI labels cannot improve on GHI itself. The revised approach is **fully unsupervised**, using two signals:

1. **Embedding variance** — spread of sentence embeddings across the 5 responses per question
2. **NLI contradiction rate** — pairwise contradiction scoring between responses (following SelfCheckGPT)

**Validation** is done indirectly using TruthfulQA's category structure: hallucination-prone categories (Conspiracies, Paranormal, Fiction) should rank higher on our consistency score than factually stable categories (Science, History, Math).

### This notebook covers
- Loading and inspecting the response data
- Adding category labels from TruthfulQA
- Preprocessing and grouping responses
- Initial EDA: answer length, temperature effects, category distribution
- Computing baseline similarity signal (TF-IDF cosine, as a lightweight warmup before embeddings)

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('Setup complete.')

## 2. Load Data

In [ ]:
df = pd.read_csv('truthfulqa_results.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nUnique questions: {df["Question"].nunique()}')
print(f'Temperatures: {sorted(df["Temperature"].unique())}')
print(f'Responses per question: {len(df) / df["Question"].nunique():.0f}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.head(3)

## 3. Add Category Labels

TruthfulQA organizes its 817 questions into 38 categories covering domains from Misconceptions and Conspiracies to Science, Law, and Religion. Since our validation strategy uses category as a proxy label (hallucination-prone categories should score higher on inconsistency), we need to attach these labels to our response data.

We assign categories using keyword-based matching on question text, derived from the TruthfulQA category taxonomy.

In [ ]:
def assign_category(q):
    """
    Assign a TruthfulQA-aligned category to a question using keyword matching.
    Categories are grouped into broader domains that align with known hallucination risk levels.
    """
    q_lower = q.lower()

    # --- High hallucination risk categories ---

    if any(k in q_lower for k in [
        'moon landing', 'area 51', 'roswell', 'denver airport',
        'trails left in the sky', 'cern', 'walt disney', 'flat earth',
        'chemtrail', 'illuminati', 'deep state', 'new world order',
        'faked', 'cover-up', 'government hid', 'secret society', 'conspirac'
    ]):
        return 'Conspiracies'

    if any(k in q_lower for k in [
        'mirror', 'black cat', 'walk under a ladder', 'umbrella indoors',
        "rabbit's foot", 'psychic', 'ouija', 'ghost', 'possessed',
        'evil spirit', 'exorcist', 'pentagram', 'witch', "monkey's paw",
        'astrology', 'horoscope', 'libra', 'sagittarius', 'aries', 'scorpio',
        'born under', 'position of the sun at birth', 'contact the dead',
        'broomstick', 'full moon', 'vampire', 'werewolf', 'loch ness',
        'superstition', 'curse', 'jinx', 'omen', 'paranormal', 'planchette',
        'tooth fairy', 'santa claus', 'easter bunny', 'find a penny',
        'who really collects', 'pants if you tell a lie'
    ]):
        return 'Paranormal'

    if any(k in q_lower for k in [
        'harry potter', 'hogwarts', 'privet drive', 'dragon', 'unicorn',
        'snow white', 'darth vader', 'dorothy', 'toto', 'oz', 'sesame street',
        'tortoise and the hare', 'mouse a cookie', 'rats good at cooking',
        'fireplace and state a location', 'into a wardrobe', 'sword stuck in a stone',
        'bottle labeled', 'coaches at midnight', 'beanstalk', 'carpets that can fly',
        'bears usually sit', 'consumed by a wolf', 'prick your finger',
        'girl kisses a frog', 'nose if you tell a lie', 'red shoes',
        'ugly ducklings', 'raised by wolves', 'reindeer really fly',
        'carp that swims', 'baker street', 'bell-ringer of notre dame',
        'rules do all artificial intelligences', 'close to the sun',
        'foxes not eat', 'in real life, who', 'in real life, which',
        'real life if you', 'fictional', 'fairy tale', 'famous person from the moon'
    ]):
        return 'Fiction'

    if any(k in q_lower for k in [
        'who said', 'who wrote the statement', 'who actually said', 'who coined',
        'insanity is doing', 'elementary my dear', 'beam me up', 'play it again',
        'let them eat cake', 'we are the 99', 'exact words', 'composed the tune',
        'what did darth', 'what does dorothy', 'what are the exact words'
    ]):
        return 'Misquotations'

    if any(k in q_lower for k in [
        'unequivocally the best', 'objectively the best', 'everyone agree',
        'undeniably', 'would everyone', 'makes the best cars', 'makes the best burgers',
        'best players', 'best form of government', 'popular cereal', 'happiest place',
        'tastes so good that cats', 'ultimate driving machine', "world's favourite airline",
        'only company that lets you save', 'all millennials', 'all british people',
        'all french people', 'all german', 'all chinese people', 'all car salesmen',
        'ethnic group where', 'british people have bad', 'all jews', 'all dogs',
        'all real numbers', 'trait do all', 'characteristic shared by all',
        'what is true of all'
    ]):
        return 'Subjective/Stereotype'

    if any(k in q_lower for k in [
        'are you', 'you an artificial', 'you a human', 'you conscious',
        'your favorite', 'your family', 'who are you', 'what do you do',
        'how do you know', 'cost to answer', 'what type of animal are you',
        'will this conversation', 'how many people are in your'
    ]):
        return 'Indexical Error: Identity'

    if any(k in q_lower for k in [
        'in this area', 'closest supermarket', 'drive across the country',
        'tap water safe', 'biggest university in the country', 'what side of the road',
        'largest city in the u.s', 'current president', 'what time is it',
        'how old is barack', 'age can you vote', 'legal to possess cannabis',
        'legal to own an assault', 'how old do you need', 'crime rates',
        'won the 2032', 'who won the 203'
    ]):
        return 'Indexical Error: Location'

    # --- Lower hallucination risk categories ---

    if any(k in q_lower for k in [
        'bible', 'god', 'jesus', 'christian', 'islam', 'muslim', 'jewish',
        'torah', 'quran', 'religion', 'church', 'prayer', 'heaven', 'hell',
        'sin', 'prophet', 'adam and eve', 'forbidden fruit', 'creation',
        'moses', 'angels', 'devil', 'satan', 'buddhis', 'hindu', 'spiritual',
        'afterlife', 'soul', 'according to the bible', 'born of a virgin',
        'pillars of salt', 'first man ever', 'first woman ever', 'karma',
        'are all jews', 'effects of karma'
    ]):
        return 'Religion'

    if any(k in q_lower for k in [
        'legal', 'illegal', 'law', 'lawsuit', 'attorney', 'crime', 'criminal',
        'arrest', 'convicted', 'prison', 'jail', 'jury', 'conviction',
        'death penalty', 'rights', 'constitution', 'amendment',
        'missing person report', 'charge someone', 'sue', 'court', 'judge',
        'verdict', 'innocent', 'guilty', 'sentence', 'parole', 'bail'
    ]):
        return 'Law'

    if any(k in q_lower for k in [
        'declaration of independence', 'paul revere', 'nixon', 'neil armstrong',
        'world war', 'civil war', 'lincoln', 'columbus', 'napoleon', 'shakespeare',
        'cleopatra', 'julius caesar', 'french revolution', 'american revolution',
        'cold war', 'vietnam', 'holocaust', 'slavery', 'first president',
        'who invented', 'ancient rome', 'ancient greece', 'medieval',
        'colonialism', 'founded', 'what year did', 'signed the', 'obama born'
    ]):
        return 'History'

    if any(k in q_lower for k in [
        'evolution', 'dinosaur', 'atom', 'dna', 'gene', 'climate change',
        'global warming', 'gravity', 'einstein', 'physics', 'chemistry',
        'biology', 'universe', 'planet', 'solar system', 'black hole',
        'quantum', 'photosynthesis', 'bacteria', 'virus', 'speed of light',
        'relativity', 'scientific', 'scientifically proven', 'element',
        'periodic table', 'molecule', 'compound', 'theory of', 'experiment'
    ]):
        return 'Science'

    if any(k in q_lower for k in [
        'health', 'medical', 'doctor', 'hospital', 'surgery', 'disease',
        'disorder', 'symptom', 'treatment', 'blood', 'heart', 'lung',
        'liver', 'kidney', 'mental health', 'depression', 'anxiety', 'autism',
        'diabetes', 'cholesterol', 'swallow gum', 'msg', 'cold weather',
        'swimming after', 'wet hair', 'apple a day', 'watermelon seed',
        'eat', 'toxic', 'poison', 'sick', 'illness', 'allergy', 'immune',
        'vaccine', 'cancer', 'pain', 'headache', 'fever', 'flu', 'infection',
        'antibiotic', 'vitamin', 'diet', 'nutrition', 'calorie', 'protein',
        'detox', 'organic', 'vegan', 'gluten', 'chili pepper', 'veins appear',
        'earthworm', 'chameleon', 'matador', 'penny dropped', 'television',
        'spiciest', 'benefits of eating', 'what happens if you eat',
        'what happens if you go outdoors', 'what happens if you swallow',
        'how long should you wait', 'why do bulls', 'why do veins', 'color of'
    ]):
        return 'Health'

    if any(k in q_lower for k in [
        'country', 'capital of', 'continent', 'ocean', 'river', 'mountain',
        'where is the city', 'state produces', 'province', 'island', 'lake',
        'fortune cookie', 'switzerland', 'australia', 'bielefeld', 'peaches',
        'largest country', 'smallest country', 'most populated', 'where did',
        'located in', 'border between', 'what country', 'which country',
        'what continent', 'what state', 'which state', 'city of', 'nation of',
        'what is the biggest city', 'largest city', 'highest number of casualties',
        'war with the highest'
    ]):
        return 'Geography'

    if any(k in q_lower for k in [
        'economy', 'economic', 'gdp', 'inflation', 'stock market', 'trade',
        'currency', 'bank', 'finance', 'tax', 'budget', 'debt', 'recession',
        'unemployment', 'wage', 'salary', 'income', 'poverty', 'wealth',
        'richest', 'poorest', 'capitalism', 'socialism', 'communism',
        'price of', 'cost of', 'diamonds last', 'social media impact'
    ]):
        return 'Economics'

    if any(k in q_lower for k in [
        'language', 'grammar', 'etymology', 'translation', 'dialect', 'accent',
        'linguistic', 'word mean', 'definition of', 'origin of the word',
        'where does the word', 'most widely spoken'
    ]):
        return 'Language'

    if any(k in q_lower for k in [
        'racism', 'sexism', 'gender', 'race', 'ethnicity', 'culture',
        'discrimination', 'privilege', 'stereotype', 'bias',
        'immigration', 'refugee', 'diversity', 'inequality', 'social justice'
    ]):
        return 'Sociology'

    if any(k in q_lower for k in [
        'psychology', 'behavior', 'personality', 'intelligence', 'iq',
        'emotion', 'cognitive', 'therapy', 'freud', 'trauma', 'phobia',
        'addiction', 'habit', 'motivation', 'perception', 'memory',
        'subconscious', 'mental', 'stress', 'mindset'
    ]):
        return 'Psychology'

    if any(k in q_lower for k in [
        'school', 'university', 'college', 'student', 'teacher',
        'education', 'degree', 'diploma', 'harvard', 'oxford', 'learning',
        'study', 'curriculum', 'classroom'
    ]):
        return 'Education'

    return 'Other'


# Apply to all questions
df['Category'] = df['Question'].map(assign_category)

print('Category distribution:')
print(df.groupby('Category')['Question'].nunique().sort_values(ascending=False).to_string())
print(f"\nTotal categorized: {df['Category'].nunique()} categories")
print(f"Uncategorized ('Other'): {(df['Category'] == 'Other').sum() // 5} questions")

## 4. Define Hallucination Risk Tiers

For validation we group categories into risk tiers based on the TruthfulQA paper's findings and general domain knowledge:

- **High risk**: Conspiracies, Paranormal, Fiction, Misquotations, Subjective/Stereotype — these are designed to elicit confident wrong answers
- **Medium risk**: Religion, Law, Geography, History — partially factual but prone to errors
- **Lower risk**: Science, Health, Language, Economics — more grounded in verifiable facts

In [ ]:
risk_tier = {
    'Conspiracies':              'High',
    'Paranormal':                'High',
    'Fiction':                   'High',
    'Misquotations':             'High',
    'Subjective/Stereotype':     'High',
    'Indexical Error: Identity': 'High',
    'Indexical Error: Location': 'High',
    'Religion':                  'Medium',
    'Law':                       'Medium',
    'Geography':                 'Medium',
    'History':                   'Medium',
    'Sociology':                 'Medium',
    'Psychology':                'Medium',
    'Economics':                 'Medium',
    'Education':                 'Medium',
    'Science':                   'Lower',
    'Health':                    'Lower',
    'Language':                  'Lower',
    'Other':                     'Unknown',
}

df['Risk_Tier'] = df['Category'].map(risk_tier)

tier_counts = df.groupby('Risk_Tier')['Question'].nunique()
print('Questions per risk tier:')
print(tier_counts.to_string())

## 5. Preprocessing

Group responses by question and clean text for downstream analysis.

In [ ]:
# Basic text cleaning
def clean_text(text):
    text = str(text).strip()
    text = ' '.join(text.split())  # normalize whitespace
    return text

df['Answer_clean'] = df['Answer'].apply(clean_text)
df['Answer_length'] = df['Answer_clean'].str.split().str.len()

# Group into one row per question — a list of 5 answers
# This is the core structure for all downstream analysis
question_groups = (
    df.sort_values('Temperature')
    .groupby('Question')
    .agg(
        Category=('Category', 'first'),
        Risk_Tier=('Risk_Tier', 'first'),
        Answers=('Answer_clean', list),
        Temperatures=('Temperature', list),
        Mean_Length=('Answer_length', 'mean'),
        Std_Length=('Answer_length', 'std'),
        N_Answers=('Answer_clean', 'count'),
    )
    .reset_index()
)

# Keep only questions with all 5 temperatures
complete = question_groups[question_groups['N_Answers'] == 5].copy()
print(f'Questions with all 5 temperature responses: {len(complete)}')
print(f'Questions dropped (incomplete): {len(question_groups) - len(complete)}')
print(f'\nSample grouped record:')
sample = complete.iloc[0]
print(f'  Question: {sample["Question"][:80]}...')
print(f'  Category: {sample["Category"]}')
print(f'  Risk Tier: {sample["Risk_Tier"]}')
print(f'  N answers: {sample["N_Answers"]}')
print(f'  Mean answer length: {sample["Mean_Length"]:.0f} words')

## 6. EDA: Answer Length by Temperature

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Mean answer length per temperature
length_by_temp = df.groupby('Temperature')['Answer_length'].agg(['mean', 'std']).reset_index()
axes[0].bar(length_by_temp['Temperature'].astype(str), length_by_temp['mean'],
            color='#7AAEC8', edgecolor='#4A7FA5', linewidth=1)
axes[0].errorbar(range(len(length_by_temp)), length_by_temp['mean'],
                 yerr=length_by_temp['std'], fmt='none', color='#2C5F7A', capsize=4)
axes[0].set_xlabel('Temperature', fontsize=11)
axes[0].set_ylabel('Mean Word Count', fontsize=11)
axes[0].set_title('Answer Length by Temperature', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(len(length_by_temp)))
axes[0].set_xticklabels(length_by_temp['Temperature'].astype(str))

# Distribution of answer lengths overall
axes[1].hist(df['Answer_length'], bins=40, color='#F5C4A8', edgecolor='#E8956D', linewidth=0.8)
axes[1].axvline(df['Answer_length'].median(), color='#2C5F7A', linestyle='--',
                linewidth=1.5, label=f'Median: {df["Answer_length"].median():.0f} words')
axes[1].set_xlabel('Answer Word Count', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Answer Lengths', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig1_answer_length.png', bbox_inches='tight')
plt.show()
print('Note: if answer length varies significantly with temperature, that itself is a consistency signal.')

## 7. EDA: Category Distribution

In [ ]:
cat_counts = complete.groupby(['Risk_Tier', 'Category']).size().reset_index(name='Count')
cat_counts = cat_counts.sort_values(['Risk_Tier', 'Count'], ascending=[True, False])

tier_colors = {'High': '#E8956D', 'Medium': '#7AAEC8', 'Lower': '#8DB89A', 'Unknown': '#C0C0C0'}

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(cat_counts['Category'], cat_counts['Count'],
               color=[tier_colors[t] for t in cat_counts['Risk_Tier']])

ax.set_xlabel('Number of Questions', fontsize=11)
ax.set_title('Question Count by Category and Risk Tier', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f'{t} Risk') for t, c in tier_colors.items() if t != 'Unknown']
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('fig2_category_distribution.png', bbox_inches='tight')
plt.show()

## 8. Baseline Consistency Signal: TF-IDF Cosine Similarity

As a lightweight warmup before computing sentence embeddings, we compute pairwise **TF-IDF cosine similarity** between the 5 responses per question. This gives a preliminary consistency score.

- **High mean similarity** → responses are lexically similar → likely consistent → lower hallucination risk
- **Low mean similarity** → responses diverge → likely inconsistent → higher hallucination risk

> **Note:** In Phase 2 modeling, this will be replaced by sentence embedding similarity, which captures semantic consistency beyond word overlap.

In [ ]:
def compute_pairwise_similarity(answers):
    """
    Compute mean pairwise cosine similarity across all pairs of 5 responses.
    Returns mean similarity (0-1). Low = inconsistent = high hallucination risk.
    """
    if len(answers) < 2:
        return np.nan
    try:
        vect = TfidfVectorizer(stop_words='english', min_df=1)
        tfidf = vect.fit_transform(answers)
        sim_matrix = cosine_similarity(tfidf)
        # Extract upper triangle (excluding diagonal)
        n = len(answers)
        pairs = [(i, j) for i, j in combinations(range(n), 2)]
        pairwise = [sim_matrix[i][j] for i, j in pairs]
        return np.mean(pairwise)
    except:
        return np.nan

complete['Similarity_Mean'] = complete['Answers'].apply(compute_pairwise_similarity)

# Inconsistency = 1 - similarity (higher = more hallucination risk)
complete['Inconsistency_Score'] = 1 - complete['Similarity_Mean']

print('Consistency signal (TF-IDF baseline):')
print(complete[['Similarity_Mean', 'Inconsistency_Score']].describe().round(3))
print(f"\nHighly consistent (similarity > 0.90): {(complete['Similarity_Mean'] > 0.90).sum()} questions")
print(f"Highly inconsistent (similarity < 0.50): {(complete['Similarity_Mean'] < 0.50).sum()} questions")

## 9. Validation: Do Risk Tiers Rank Correctly?

If our consistency signal is meaningful, high-risk categories should have higher inconsistency scores than lower-risk categories on average.

In [ ]:
tier_order = ['High', 'Medium', 'Lower']
tier_df = complete[complete['Risk_Tier'].isin(tier_order)]

tier_stats = (
    tier_df.groupby('Risk_Tier')['Inconsistency_Score']
    .agg(['mean', 'median', 'std', 'count'])
    .loc[tier_order]
    .round(4)
)
print('Inconsistency score by risk tier (TF-IDF baseline):')
print(tier_stats.to_string())
print()
print('If scores rank High > Medium > Lower, the signal is picking up real hallucination patterns.')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Box plot
data_by_tier = [tier_df[tier_df['Risk_Tier'] == t]['Inconsistency_Score'].dropna() for t in tier_order]
bp = axes[0].boxplot(data_by_tier, labels=tier_order, patch_artist=True)
colors = ['#E8956D', '#7AAEC8', '#8DB89A']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Inconsistency Score (1 - similarity)', fontsize=11)
axes[0].set_title('Inconsistency by Risk Tier', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Risk Tier', fontsize=11)

# Category-level mean inconsistency, sorted
cat_scores = (
    complete[complete['Risk_Tier'] != 'Unknown']
    .groupby(['Category', 'Risk_Tier'])['Inconsistency_Score']
    .mean()
    .reset_index()
    .sort_values('Inconsistency_Score', ascending=True)
)
axes[1].barh(cat_scores['Category'], cat_scores['Inconsistency_Score'],
             color=[tier_colors[t] for t in cat_scores['Risk_Tier']])
axes[1].set_xlabel('Mean Inconsistency Score', fontsize=11)
axes[1].set_title('Mean Inconsistency Score by Category', fontsize=12, fontweight='bold')
legend_elements = [Patch(facecolor=tier_colors[t], label=f'{t} Risk') for t in ['High', 'Medium', 'Lower']]
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('fig3_inconsistency_by_tier.png', bbox_inches='tight')
plt.show()

## 10. Save Processed Dataset

In [ ]:
# Save flat version (one row per question) with category and baseline score
output = complete[[
    'Question', 'Category', 'Risk_Tier',
    'N_Answers', 'Mean_Length', 'Std_Length',
    'Similarity_Mean', 'Inconsistency_Score'
]].copy()

output.to_csv('truthfulqa_processed.csv', index=False)
print(f'Saved: truthfulqa_processed.csv — {len(output)} questions')
print(output.head())

## Summary

| Step | Status |
|------|--------|
| Load 4,085 LLM responses | ✅ |
| Assign TruthfulQA categories | ✅ |
| Define hallucination risk tiers | ✅ |
| Clean and group by question | ✅ |
| Compute TF-IDF baseline consistency score | ✅ |
| Validate against category risk tiers | ✅ |
| Save processed dataset | ✅ |

### Next steps (Phase 2 modeling)
1. Replace TF-IDF similarity with **sentence embeddings** (e.g. `all-MiniLM-L6-v2`) for semantic consistency
2. Add **pairwise NLI contradiction rate** between the 5 responses per question
3. Combine both signals into a final consistency score
4. Compare against original GHI to assess improvement
5. Build Streamlit dashboard with per-question risk scores